# Fleet Maintenance Planning

**Team:** 38

### 1. Project Overview

#### Objective

Predict which vehicles should be inspected and determine
the appropriate spare-parts budget for each inspected vehicle.

The objective is to minimise the total maintenance cost
using the competition's specified cost function.

#### Project Workflow

1. Data loading and exploration
2. Preprocessing and feature selection
3. Failure prediction
4. Repair-cost prediction
5. Cost-based decision optimisation
6. Final submission generation

### 2. Imports and Config

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project root without changing the working directory.
current = Path.cwd().resolve()

ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data").is_dir() and (p / "src").is_dir()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate project root containing data/ and src/."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Import all shared paths and settings from one place.
from src.config import (
    DATA_DIR,
    RESULTS_DIR,
    SUBMISSIONS_DIR,
    RANDOM_STATE,
)

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

Project root: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38
Data directory: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\data
Results directory: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\results


### 3. Data Loading:

In [2]:
from src.feature_analysis import load_data

# Load training and test datasets
train_df, test_df = load_data(DATA_DIR)

# Load sample submission for final format validation
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)
print("Sample submission:", sample_submission.shape)

Training data: (56000, 174)
Testing data: (20000, 171)
Sample submission: (20000, 3)


### 4. Data Validation

In [5]:
# Check required columns
assert "failure" in train_df.columns
assert "repair_cost" in train_df.columns
assert "id" in train_df.columns

# Check identifiers
assert train_df["id"].is_unique
assert test_df["id"].is_unique

# Check target
assert train_df["failure"].notna().all()
assert train_df["failure"].isin([0, 1]).all()

# Check feature compatibility
sensor_cols = [
    col for col in train_df.columns
    if col.startswith("s")
]

assert set(sensor_cols) == {
    col for col in test_df.columns
    if col.startswith("s")
}

print("Basic data validation passed!")

Basic data validation passed!


### 5. EDA Summary

- The failure rate is highly imbalanced (0.71%).
- Repair costs are strongly right-skewed.
- Sensor features contain substantial missing values.
- Eight sensors have more than 50% missing values.
- One sensor (`s146`) is constant.
- Several sensors show substantial train–test differences, suggesting potential distribution shift between depots.

### 6. Cross-Validation Setup

In [3]:
from src.cv_utils import load_folds, validate_folds
from src.lightgbm_model import get_base_features

# Select features available and usable at test time
features = get_base_features(train_df, test_df)

X = train_df[features]
y = train_df["failure"]

# Load and validate the shared CV folds
folds = load_folds()
validate_folds(folds, len(X))

print(f"Training vehicles: {len(X):,}")
print(f"Features selected: {len(features)}")
print(f"CV folds: {len(folds)}")
print(f"Features excluded: {train_df.shape[1] - len(features) - 3}")

Fold validation successful.
Training vehicles: 56,000
Features selected: 167
CV folds: 5
Features excluded: 4


In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("results/repair_oof.csv")

# Actual repair cost: zero for vehicles that did not fail.
actual = df["repair_cost"].fillna(0).to_numpy()
prob = df["failure_probability"].to_numpy()
folds = df["fold"].to_numpy()

budget_models = {
    "constant": "constant_budget",
    "median": "median",
    "forest": "forest",
    "xgb": "xgb",
    "lgbm": "lgbm",
}

multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]


def competition_cost(actual, budget, flags):
    inspection = (
        10
        + 0.20 * np.maximum(actual - budget, 0)
        + 0.02 * np.maximum(budget - actual, 0)
    )
    return np.where(flags, inspection, actual)


def tune_policy(data, budget_col):
    y = data["repair_cost"].fillna(0).to_numpy()
    p = data["failure_probability"].to_numpy()
    base_budget = data[budget_col].to_numpy()

    # Sort observations by descending failure probability.
    order = np.argsort(-p, kind="stable")
    sorted_p = p[order]
    sorted_y = y[order]
    sorted_budget = base_budget[order]

    best = None

    # Only consider thresholds that do not split tied probabilities.
    group_ends = np.r_[
        np.where(sorted_p[:-1] != sorted_p[1:])[0],
        len(sorted_p) - 1
    ]

    for multiplier in multipliers:
        budget = sorted_budget * multiplier

        inspection = (
            10
            + 0.20 * np.maximum(sorted_y - budget, 0)
            + 0.02 * np.maximum(budget - sorted_y, 0)
        )

        # Cost change from inspecting instead of not inspecting.
        delta = inspection - sorted_y
        cumulative = np.cumsum(delta)

        # Include the option of inspecting nobody.
        candidates = [(sorted_y.sum(), None, 0)]

        for end in group_ends:
            total = sorted_y.sum() + cumulative[end]
            candidates.append((total, sorted_p[end], end + 1))

        cost, threshold, inspected = min(candidates, key=lambda x: x[0])

        if best is None or cost < best["training_cost"]:
            best = {
                "threshold": threshold,
                "multiplier": multiplier,
                "training_cost": cost,
                "training_inspected": inspected,
            }

    return best


results = []

for model_name, budget_col in budget_models.items():
    for fold in sorted(df["fold"].unique()):
        train = df[df["fold"] != fold]
        valid = df[df["fold"] == fold]

        policy = tune_policy(train, budget_col)

        if policy["threshold"] is None:
            flags = np.zeros(len(valid), dtype=bool)
        else:
            flags = (
                valid["failure_probability"].to_numpy()
                >= policy["threshold"]
            )

        budgets = (
            valid[budget_col].to_numpy()
            * policy["multiplier"]
        )

        costs = competition_cost(
            valid["repair_cost"].fillna(0).to_numpy(),
            budgets,
            flags
        )

        results.append({
            "model": model_name,
            "fold": int(fold),
            "threshold": policy["threshold"],
            "multiplier": policy["multiplier"],
            "inspected": int(flags.sum()),
            "cost": costs.sum(),
        })


results_df = pd.DataFrame(results)

summary = (
    results_df.groupby("model")
    .agg(
        total_cost=("cost", "sum"),
        avg_cost_per_vehicle=("cost", "sum"),
        total_inspected=("inspected", "sum"),
    )
    .sort_values("total_cost")
)

summary["avg_cost_per_vehicle"] /= len(df)

print("Cross-fitted policy comparison:")
print(summary.round(3).to_string())

print("\nFold-level results:")
print(results_df.round(3).to_string(index=False))


Cross-fitted policy comparison:
          total_cost  avg_cost_per_vehicle  total_inspected
model                                                      
lgbm       82147.051                 1.467             1224
forest     82196.981                 1.468             1224
xgb        82512.466                 1.473             1224
median     82532.480                 1.474             1273
constant   85022.999                 1.518             1146

Fold-level results:
   model  fold  threshold  multiplier  inspected      cost
constant     0      0.019        0.50        290 12098.965
constant     1      0.027        0.50        190 13593.344
constant     2      0.025        0.50        219 18988.198
constant     3      0.026        0.50        231 22575.980
constant     4      0.025        0.50        216 17766.512
  median     0      0.018        1.50        293 11370.535
  median     1      0.019        1.50        229 13167.280
  median     2      0.025        1.50        219 18523.